# Customer Shopping Behavior: Data Cleaning, Feature Engineering and MySQL Load

This notebook prepares the customer shopping dataset (3,900 records, 18 columns) for SQL analysis and Power BI.

**Steps:** load -> inspect -> quick EDA -> clean -> engineer features -> save cleaned CSV -> load into MySQL

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

## 1. Load the data

In [ ]:
df = pd.read_csv("data/customer_shopping_behavior.csv")
print(df.shape)
df.head()

## 2. Inspect data quality

In [ ]:
df.info()

In [ ]:
# Missing values and duplicates
print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
df.describe()

## 3. Quick exploratory analysis

In [ ]:
for col in ["Gender", "Category", "Season", "Subscription Status", "Payment Method"]:
    print(df[col].value_counts(), "\n")

In [ ]:
# Average purchase amount by group
for col in ["Category", "Gender", "Season"]:
    print(df.groupby(col)["Purchase Amount (USD)"].mean().round(2), "\n")

## 4. Clean the data

In [ ]:
# Review Rating has 37 missing values: fill with the overall mean rating
df["Review Rating"] = df["Review Rating"].fillna(df["Review Rating"].mean())
print("Missing ratings left:", df["Review Rating"].isnull().sum())

In [ ]:
# Standardize column names (lowercase, underscores) and shorten the amount column
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df = df.rename(columns={"purchase_amount_(usd)": "purchase_amount"})
df.columns.tolist()

In [ ]:
# discount_applied and promo_code_used are identical, so drop the duplicate column
print("Identical columns:", (df["discount_applied"] == df["promo_code_used"]).all())
df = df.drop(columns="promo_code_used")

## 5. Feature engineering

In [ ]:
# Age groups with equal-sized quartiles
labels = ["Young Adult", "Adult", "Middle-aged", "Senior"]
df["age_group"] = pd.qcut(df["age"], q=4, labels=labels)
df.groupby("age_group", observed=True)["age"].agg(["min", "max", "count"])

In [ ]:
# Convert purchase frequency text into days
frequency_mapping = {
    "Weekly": 7,
    "Fortnightly": 14,
    "Bi-Weekly": 14,
    "Monthly": 30,
    "Quarterly": 90,
    "Every 3 Months": 90,
    "Annually": 365,
}
df["purchase_frequency_days"] = df["frequency_of_purchases"].map(frequency_mapping)

# Check that every value was mapped
assert df["purchase_frequency_days"].notnull().all(), "Some frequency values were not mapped"

## 6. Save the cleaned dataset

In [ ]:
df.to_csv("data/customer_shopping_behavior_cleaned.csv", index=False)
df.shape

## 7. Load into MySQL

Credentials are read from environment variables (never write a password in the notebook).
Copy `.env.example` to `.env` and fill it in, or set the variables in your terminal.

In [ ]:
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

connection_url = URL.create(
    "mysql+pymysql",
    username=os.getenv("DB_USER", "root"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST", "localhost"),
    port=int(os.getenv("DB_PORT", 3306)),
    database=os.getenv("DB_NAME", "customer_behavior"),
)
engine = create_engine(connection_url)

In [ ]:
rows = df.to_sql("customer_shopping_behavior", con=engine, if_exists="replace", index=False)
print("Rows written:", rows)

In [ ]:
# Verify the load
pd.read_sql("SELECT COUNT(*) AS total_rows FROM customer_shopping_behavior", con=engine)